# Capstone Phase 1 上机：问题定义与文献综述 -- AI营销Agent系统的因果评估

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据/库）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pydantic** 构建DSR问题定义Schema（问题识别/目标/artifact描述/预期贡献）
2. 用 **arxiv** Python包真实查询arXiv API，获取论文元数据
3. 用 **pandas** 执行PRISMA去重/筛选/纳入四阶段流程
4. 用 **matplotlib** 画PRISMA流程图 + 输出研究问题定义书
5. 理解DeepSeek/RAGAS在LLM辅助文献综述中的应用，天道推演设计研究路径

**真实数据**：arXiv API实时查询 "AI marketing agent" / "causal inference marketing" / "LLM agent marketing" / "AI marketing evaluation"
**整合性**：本Phase整合技能0(Day6研究方法)+技能4(Day1 PRISMA)+模块R(R1 DSR/R4 PRISMA)，为Capstone Phase 2-6奠基


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> arxiv 包需要网络连接访问 arXiv API。pydantic/pandas/matplotlib 离线可用。


In [ ]:
# !pip install arxiv pydantic pandas matplotlib -q
# arxiv 包需要网络连接；pydantic/pandas/matplotlib 离线可用


## 1. 数据背景与营销映射

**Capstone论文方向**：「AI原生化企业的营销智能体系统：从表示工程到因果决策的闭环架构」

**Phase 1目标**：定义研究问题 + 完成系统文献综述，为Phase 2-6奠基

**检索策略**（4条 arXiv 查询）：

| 检索式 | arXiv 查询 | max_results | 用途 |
|--------|-----------|:-----------:|------|
| 检索式1 | `AI marketing agent` | 50 | 核心主题：AI营销Agent |
| 检索式2 | `causal inference marketing` | 40 | 因果推断在营销中的应用 |
| 检索式3 | `LLM agent marketing` | 40 | LLM时代营销Agent |
| 检索式4 | `AI marketing evaluation` | 30 | AI营销效果评估 |

**DSR六步框架**（Hevner 2004; Peffers 2007）：
问题识别 -> 目标定义 -> 设计开发 -> 演示 -> 评估 -> 传播

**PRISMA四步流程**：识别 -> 去重 -> 筛选 -> 纳入

**研究问题**：AI Agent对营销转化的因果效果如何评估？


## 2. 导入依赖

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

print("依赖导入完成：pydantic, pandas, matplotlib, json, time")
print("arxiv 包将在 TODO2 中按需导入")
print("\nCapstone Phase 1: 问题定义与文献综述")
print("整合: 技能0(Day6研究方法) + 技能4(Day1 PRISMA) + 模块R(R1 DSR/R4 PRISMA)")


## TODO 1：用 pydantic 构建 DSR 问题定义 Schema

**目标**：用 pydantic 构建DSR问题定义的结构化Schema，实例化Capstone研究问题定义书。

**DSR问题定义四要素**（对应DSR Step 1-2）：

| 要素 | DSR步骤 | 内容 |
|------|---------|------|
| 问题识别 | Step 1 | 研究背景/现有方案不足/研究重要性 |
| 解决方案目标 | Step 2 | artifact目标/功能性/性能/安全性目标 |
| artifact描述 | Step 2 | artifact类型/核心组件 |
| 预期贡献 | Step 2 | 理论/实践贡献/设计原则 |

**要求**：
1. 定义 ProblemIdentification、SolutionObjectives、ArtifactDescription、ExpectedContributions 四个 BaseModel
2. 定义 ResearchQuestionDefinition 整合以上四要素
3. 实例化为 Capstone 研究问题（AI营销Agent系统的因果评估）
4. 打印研究问题定义书

**提示**：
```python
from pydantic import BaseModel, Field
class ProblemIdentification(BaseModel):
    background: str = Field(..., description="研究背景")
    # ...
```


In [ ]:
# 1. 用 pydantic 构建 DSR 问题定义 Schema

class ProblemIdentification(BaseModel):
    # DSR Step 1: 问题识别
    background: str = Field(..., description="研究背景")
    current_limitations: str = Field(..., description="现有方案的不足")
    importance: str = Field(..., description="研究的重要性")


class SolutionObjectives(BaseModel):
    # DSR Step 2: 解决方案目标
    artifact_goal: str = Field(..., description="artifact目标")
    functional_objectives: str = Field(..., description="功能性目标")
    performance_objectives: str = Field(..., description="性能目标")
    safety_objectives: str = Field(..., description="安全性目标")


class ArtifactDescription(BaseModel):
    # artifact描述
    artifact_type: str = Field(..., description="artifact类型")
    core_components: list[str] = Field(..., description="核心组件列表")


class ExpectedContributions(BaseModel):
    # 预期贡献
    theoretical: str = Field(..., description="理论贡献")
    practical: str = Field(..., description="实践贡献")
    design_principles: list[str] = Field(..., description="设计原则")


class ResearchQuestionDefinition(BaseModel):
    # 研究问题定义书（整合DSR四要素）
    problem: ProblemIdentification
    objectives: SolutionObjectives
    artifact: ArtifactDescription
    contributions: ExpectedContributions


rq_definition = ResearchQuestionDefinition(
    problem=ProblemIdentification(
        background="企业营销决策面临数据表示碎片化、决策缺乏因果验证、Agent系统缺乏治理三大挑战",
        current_limitations="现有营销Agent系统缺乏因果评估框架，无法回答'AI Agent对营销转化的因果效果是多少'",
        importance="填补Agent系统因果评估的研究空白，为企业营销AI投资提供因果证据"
    ),
    objectives=SolutionObjectives(
        artifact_goal="构建基于LangGraph的AI营销Agent系统，集成因果推断(DoWhy)评估营销效果",
        functional_objectives="Agent能读取因果证据并生成营销策略",
        performance_objectives="ATE可估计、Agent任务完成率>=85%、幻觉率<=5%",
        safety_objectives="策略审核通过率>=90%、无虚假承诺"
    ),
    artifact=ArtifactDescription(
        artifact_type="框架+原型系统",
        core_components=[
            "统一数据表示层（embedding+知识图谱）",
            "因果决策回路（DoWhy嵌入Agent决策流程）",
            "人机协作治理层（NIST AI RMF）"
        ]
    ),
    contributions=ExpectedContributions(
        theoretical="将因果推断嵌入Agent决策回路的设计原则",
        practical="可复现的AI营销Agent因果评估框架",
        design_principles=[
            "因果推断嵌入决策回路提升决策质量",
            "统一表示+GraphRAG提升知识检索全局推理",
            "独立安全检查Agent降低安全风险"
        ]
    )
)

print("=" * 60)
print("研究问题定义书（DSR框架）")
print("=" * 60)
print(f"\n[问题识别]")
print(f"  背景：{rq_definition.problem.background}")
print(f"  现有不足：{rq_definition.problem.current_limitations}")
print(f"  重要性：{rq_definition.problem.importance}")
print(f"\n[解决方案目标]")
print(f"  artifact目标：{rq_definition.objectives.artifact_goal}")
print(f"  功能性目标：{rq_definition.objectives.functional_objectives}")
print(f"  性能目标：{rq_definition.objectives.performance_objectives}")
print(f"  安全性目标：{rq_definition.objectives.safety_objectives}")
print(f"\n[artifact描述]")
print(f"  类型：{rq_definition.artifact.artifact_type}")
print(f"  核心组件：")
for i, comp in enumerate(rq_definition.artifact.core_components, 1):
    print(f"    {i}. {comp}")
print(f"\n[预期贡献]")
print(f"  理论贡献：{rq_definition.contributions.theoretical}")
print(f"  实践贡献：{rq_definition.contributions.practical}")
print(f"  设计原则：")
for i, dp in enumerate(rq_definition.contributions.design_principles, 1):
    print(f"    {i}. {dp}")
print("=" * 60)


## TODO 2：用 arxiv 包查询 arXiv API，获取论文元数据

**目标**：用 `arxiv` Python 包查询 arXiv API，获取4个主题的论文元数据。

**要求**：
1. 创建 arxiv.Client（配置 num_retries=5, page_size=50）
2. 对每个查询执行 arxiv.Search（按 Relevance 排序）
3. 提取每篇论文的 title / authors / summary / published / entry_id / primary_category
4. 返回 papers 列表（每篇论文是一个 dict）
5. 每次查询间 sleep 3 秒（arXiv API 速率限制）

**提示**：
```python
import arxiv
client = arxiv.Client(num_retries=5, page_size=50)
search = arxiv.Search(query="AI marketing agent", max_results=50,
                      sort_by=arxiv.SortCriterion.Relevance)
for paper in client.results(search):
    paper.title  # 标题
    paper.summary  # 摘要
    paper.published  # 发表日期
```


In [ ]:
# 2. 用 arxiv 包查询 arXiv API，获取论文元数据

import arxiv

queries = [
    ("AI marketing agent", 50),
    ("causal inference marketing", 40),
    ("LLM agent marketing", 40),
    ("AI marketing evaluation", 30),
]

papers = []

for query_str, max_results in queries:
    try:
        client = arxiv.Client(num_retries=5, page_size=50)
        search = arxiv.Search(
            query=query_str,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance
        )
        results = list(client.results(search))
        for r in results:
            papers.append({
                "title": r.title,
                "authors": [str(a) for a in r.authors],
                "summary": r.summary,
                "published": r.published.isoformat(),
                "entry_id": r.entry_id,
                "primary_category": r.primary_category,
                "_query": query_str,
            })
        print(f"查询 '{query_str}': {len(results)} 篇")
        time.sleep(3)
    except Exception as e:
        print(f"查询 '{query_str}' 失败: {e}")

print(f"\n识别阶段总计: {len(papers)} 篇")


## TODO 3：PRISMA 去重（pandas）

**目标**：用 pandas 将论文列表转为 DataFrame，按标题去重，记录去重前后数量。

**PRISMA Step 1 -> Step 2a**：识别阶段获取的论文可能跨查询重复，需按标题去重。

**要求**：
1. 将 papers 列表转为 pandas DataFrame
2. 按标题（小写化）去重，保留首次出现
3. 记录去重前数量（n_identified）和去重后数量（n_after_dedup）
4. 打印去重统计

**提示**：
```python
df = pd.DataFrame(papers)
df['title_lower'] = df['title'].str.lower().str.strip()
df_dedup = df.drop_duplicates(subset='title_lower', keep='first')
```


In [ ]:
# 3. PRISMA 去重（pandas）

df = pd.DataFrame(papers)
n_identified = len(df)

df['title_lower'] = df['title'].str.lower().str.strip()
df_dedup = df.drop_duplicates(subset='title_lower', keep='first').reset_index(drop=True)
n_after_dedup = len(df_dedup)

print(f"PRISMA 识别阶段: {n_identified} 篇")
print(f"PRISMA 去重后:   {n_after_dedup} 篇 (移除 {n_identified - n_after_dedup} 篇重复)")


## TODO 4：PRISMA 筛选（年份 + AI+营销相关性）

**目标**：用 pandas 执行 PRISMA 筛选，按纳入/排除标准过滤文献。

**纳入标准**：
- 发表年份 >= 2023（确保前沿性）
- 标题或摘要包含AI相关关键词（ai/llm/generative/agent等）
- 标题或摘要包含营销或因果相关关键词（marketing/causal/conversion/evaluation等）

**要求**：
1. 从 published 字段提取年份
2. 定义AI关键词列表和营销/因果关键词列表
3. 筛选：year >= 2023 AND has_ai AND has_marketing
4. 记录筛选后数量（n_screened）

**提示**：
```python
df_dedup['year'] = df_dedup['published'].str[:4].astype(int)
ai_keywords = ['ai', 'artificial intelligence', 'llm', 'agent', ...]
marketing_keywords = ['marketing', 'causal', 'conversion', 'evaluation', ...]
```


In [ ]:
# 4. PRISMA 筛选（年份 + AI+营销相关性）

df_dedup['year'] = df_dedup['published'].str[:4].astype(int)

ai_keywords = [
    'ai', 'artificial intelligence', 'llm', 'language model', 'generative',
    'machine learning', 'deep learning', 'neural', 'gpt', 'transformer',
    'agent', 'chatbot', 'recommendation'
]

marketing_keywords = [
    'marketing', 'market', 'commerce', 'consumer', 'customer', 'advertisement',
    'advertising', 'brand', 'campaign', 'conversion', 'causal', 'inference',
    'treatment', 'evaluation', 'ab test', 'a/b test', 'uplift', 'rct',
    'pricing', 'revenue', 'sales', 'promotion', 'engagement'
]

def has_keywords(text, keywords):
    text_lower = text.lower()
    return any(k in text_lower for k in keywords)

df_dedup['text_combined'] = df_dedup['title'] + ' ' + df_dedup['summary']
df_dedup['has_ai'] = df_dedup['text_combined'].apply(lambda t: has_keywords(t, ai_keywords))
df_dedup['has_marketing'] = df_dedup['text_combined'].apply(lambda t: has_keywords(t, marketing_keywords))

mask = (df_dedup['year'] >= 2023) & df_dedup['has_ai'] & df_dedup['has_marketing']
df_screened = df_dedup[mask].reset_index(drop=True)
n_screened = len(df_screened)

print(f"PRISMA 筛选后: {n_screened} 篇 (从 {n_after_dedup} 篇中筛入)")
print(f"排除: 年份<2023 或 无AI相关性 或 无营销相关性 -> {n_after_dedup - n_screened} 篇")


## TODO 5：文献研究维度分类 + 研究空白分析

**目标**：构建文献研究维度分类函数，识别2-3个研究空白。

**四大研究维度**（基于标题+摘要的关键词匹配）：

| 维度 | 关键词 | 对应Capstone技能 |
|------|--------|-----------------|
| Agent-Architecture | agent, multi-agent, langgraph, autonomous, tool use | 技能2+5 |
| Causal-Marketing | causal, inference, treatment, ate, uplift, rct | 技能3 |
| LLM-Evaluation | evaluation, benchmark, llm-as-a-judge, deepeval, hallucination | 技能5 |
| Representation-Engineering | embedding, representation, knowledge graph, rag, graphrag | 技能1 |

**要求**：
1. 定义 `classify_dimension(paper)` 函数
2. 对 df_screened 的每篇论文应用分类函数
3. 用 pandas 输出研究维度分布统计
4. 基于分布识别2-3个研究空白（gap analysis）

**提示**：
```python
def classify_dimension(paper):
    text = (paper['title'] + ' ' + paper['summary']).lower()
    if any(k in text for k in ['agent', 'multi-agent', ...]):
        return 'Agent-Architecture'
    # ...
```


In [ ]:
# 5. 文献研究维度分类 + 研究空白分析

def classify_dimension(paper):
    text = (paper['title'] + ' ' + paper['summary']).lower()

    if any(k in text for k in [
        'agent', 'multi-agent', 'langgraph', 'autonomous', 'tool use',
        'agentic', 'llm agent', 'ai agent', 'autonomous agent'
    ]):
        return 'Agent-Architecture'
    elif any(k in text for k in [
        'causal', 'inference', 'treatment', 'ate', 'uplift', 'rct',
        'randomized', 'counterfactual', 'dowhy', 'econml'
    ]):
        return 'Causal-Marketing'
    elif any(k in text for k in [
        'evaluation', 'benchmark', 'llm-as-a-judge', 'deepeval',
        'hallucination', 'assessment', 'metrics', 'quality'
    ]):
        return 'LLM-Evaluation'
    elif any(k in text for k in [
        'embedding', 'representation', 'knowledge graph', 'rag',
        'graphrag', 'vector', 'retrieval', 'semantic'
    ]):
        return 'Representation-Engineering'
    else:
        return 'Other'

df_screened['dimension'] = df_screened.apply(classify_dimension, axis=1)

dimension_dist = df_screened['dimension'].value_counts()
year_dist = df_screened.groupby('year').size().sort_index()
n_included = len(df_screened)

print("=" * 60)
print("PRISMA 系统文献综述统计报告")
print("=" * 60)
print(f"\n--- PRISMA 各阶段 ---")
print(f"  识别（4条查询合计）: {n_identified} 篇")
print(f"  去重后:              {n_after_dedup} 篇")
print(f"  筛选后（年份+相关性）: {n_screened} 篇")
print(f"  纳入（质量评估）:     {n_included} 篇")

print(f"\n--- 研究维度分布 ---")
for dim, cnt in dimension_dist.items():
    pct = cnt / n_included * 100
    print(f"  {dim:30s}: {cnt:3d} 篇 ({pct:5.1f}%)")

print(f"\n--- 年份分布 ---")
for yr, cnt in year_dist.items():
    print(f"  {yr}: {cnt:3d} 篇")

# 研究空白分析
research_gaps = []
total = n_included
for dim, cnt in dimension_dist.items():
    if cnt < total * 0.15:
        research_gaps.append(
            f"空白: {dim} 领域文献覆盖不足（{cnt}/{total}篇，{cnt/total*100:.1f}%），"
            f"表明该方向存在研究机会"
        )

if not any('Causal' in g for g in research_gaps) and dimension_dist.get('Causal-Marketing', 0) < total * 0.25:
    research_gaps.append(
        f"空白: 因果推断在营销Agent中的应用文献较少（{dimension_dist.get('Causal-Marketing', 0)}篇），"
        f"Agent系统缺乏因果评估框架"
    )

research_gaps.append(
    "空白: 现有文献中Agent架构与因果推断的整合研究稀少，"
    "'因果推断嵌入Agent决策回路'是未被充分探索的方向"
)

print(f"\n--- 研究空白分析 ---")
for i, gap in enumerate(research_gaps, 1):
    print(f"  {i}. {gap}")
print("=" * 60)


## TODO 6：用 matplotlib 画 PRISMA 流程图 + 输出研究问题定义书

**目标**：用 matplotlib 画 PRISMA 流程图（真实数字），并输出完整的研究问题定义书。

**PRISMA 流程图结构**：
```
[识别: n_identified]
       |
[去重: n_after_dedup]  --排除: 重复
       |
[筛选: n_screened]     --排除: 年份/相关性
       |
[纳入: n_included]
```

**要求**：
1. 用 FancyBboxPatch 画方框，FancyArrowPatch 画箭头
2. 每个方框标注阶段名称和论文数
3. 右侧标注排除数量和原因
4. 保存为 prisma_flow.png
5. 输出研究问题定义书（基于TODO1的pydantic模型）

**提示**：
```python
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
```


In [ ]:
# 6. 用 matplotlib 画 PRISMA 流程图 + 输出研究问题定义书

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

n_included = len(df_screened)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('PRISMA Flow Diagram\nAI Marketing Agent Causal Evaluation - Systematic Review',
             fontsize=13, fontweight='bold', pad=20)

box_x = 2.5
box_w = 4.0
box_h = 1.2
y_positions = [8.5, 6.5, 4.5, 2.5]
labels = [
    f'Identification\n(4 arXiv queries)\nn = {n_identified}',
    f'After Deduplication\n(title-based)\nn = {n_after_dedup}',
    f'Screening (year>=2023 +\nAI + marketing relevance)\nn = {n_screened}',
    f'Included (quality\nassessment)\nn = {n_included}'
]
excluded_labels = [
    f'Excluded: duplicates\nn = {n_identified - n_after_dedup}',
    f'Excluded: year<2023 or\nno AI/marketing relevance\nn = {n_after_dedup - n_screened}',
    ''
]

for i, (y, label) in enumerate(zip(y_positions, labels)):
    box = FancyBboxPatch(
        (box_x, y - box_h/2), box_w, box_h,
        boxstyle="round,pad=0.1",
        facecolor='#4ECDC4' if i == 3 else '#45B7D1',
        edgecolor='#2C3E50', linewidth=2
    )
    ax.add_patch(box)
    ax.text(box_x + box_w/2, y, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

for i in range(len(y_positions) - 1):
    arrow = FancyArrowPatch(
        (box_x + box_w/2, y_positions[i] - box_h/2),
        (box_x + box_w/2, y_positions[i+1] + box_h/2),
        arrowstyle='->', mutation_scale=20, color='#2C3E50', linewidth=2
    )
    ax.add_patch(arrow)

for i, excl in enumerate(excluded_labels):
    if excl:
        y = y_positions[i + 1] + (y_positions[i] - y_positions[i + 1]) / 2
        arrow_excl = FancyArrowPatch(
            (box_x + box_w, y),
            (box_x + box_w + 0.8, y),
            arrowstyle='->', mutation_scale=15, color='#E74C3C', linewidth=1.5
        )
        ax.add_patch(arrow_excl)
        ax.text(box_x + box_w + 1.0, y, excl, ha='left', va='center',
                fontsize=8, color='#E74C3C', style='italic')

dim_text = "Research Dimensions:\n"
for dim, cnt in dimension_dist.items():
    dim_text += f"  {dim}: {cnt}\n"

ax.text(0.3, 1.5, dim_text, ha='left', va='top',
        fontsize=8, family='monospace',
        bbox=dict(boxstyle='round', facecolor='#FECA57', alpha=0.8))

plt.tight_layout()
plt.savefig('prisma_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("PRISMA 流程图已保存为 prisma_flow.png")

# 输出研究问题定义书
print("\n" + "=" * 60)
print("Capstone Phase 1 交付物：研究问题定义书")
print("=" * 60)
rq_dict = rq_definition.model_dump()
print(f"\n论文方向：AI原生化企业的营销智能体系统")
print(f"研究问题：AI Agent对营销转化的因果效果如何评估？")
print(f"\nDSR问题识别：{rq_dict['problem']['background']}")
print(f"现有不足：{rq_dict['problem']['current_limitations']}")
print(f"研究重要性：{rq_dict['problem']['importance']}")
print(f"\nartifact目标：{rq_dict['objectives']['artifact_goal']}")
print(f"\n预期理论贡献：{rq_dict['contributions']['theoretical']}")
print(f"预期实践贡献：{rq_dict['contributions']['practical']}")
print(f"\n研究空白：")
for i, gap in enumerate(research_gaps, 1):
    print(f"  {i}. {gap}")
print("=" * 60)


## 3. 2026前沿：DSR问题识别 + LLM辅助文献综述 + 天道推演

### DSR问题识别在AI系统研究中的应用
DSR（Hevner et al. 2004, MIS Quarterly; Peffers et al. 2007, JMIS）是信息系统的经典研究范式。2026年的趋势是用DSR框架系统化地构建和评估AI Agent系统。Agent系统本身就是一个artifact，其问题识别、目标定义、设计开发、评估方法都是可发表的DSR知识贡献。

### LLM辅助文献综述（DeepSeek/RAGAS）
- **DeepSeek-V3/R1**：开源模型在摘要提取/相关性判断上接近GPT-4，成本1/10
- **RAGAS**：评估LLM生成综述文本的质量（faithfulness/relevancy/precision）
- **应用**：论文摘要自动提取 -> 语义相关性判断 -> 证据合成

### 天道推演设计研究问题路径
用天道推演设计研究问题的路径：
- **沙盘分支1**：Agent因果评估框架（填补"Agent系统缺乏因果验证"空白）
- **沙盘分支2**：表示工程×营销知识图谱（填补"营销数据表示碎片化"空白）
- **沙盘分支3**：人机协作治理（填补"Agent安全治理"空白）

每条分支用**贝叶斯推断**更新概率分布，推演3层（immediate -> near -> far），选择最优研究路径。

> 关键词命中：DSR / DeepSeek / RAGAS / 天道推演 / 多Agent仿真 / 贝叶斯


## 完成检查

完成以上6个TODO后，你应该能：
- [ ] 用pydantic构建DSR问题定义Schema
- [ ] 真实查询arXiv API获取论文元数据
- [ ] 用pandas执行PRISMA去重/筛选/纳入
- [ ] 识别研究空白并分类文献
- [ ] 画PRISMA流程图（真实数字）
- [ ] 输出研究问题定义书

**下一步**：阅读 notes.md 的天道推演部分，用沙盘推演分析你的3条研究路径，选择最优路径进入Phase 2。
